# NoisiQ — Week 5 Notebook
**Noise-Aware Quantum Circuit Simulation and Visualization**
*Period: April 20 – April 26, 2026*

---

## Purpose of This Notebook

This notebook serves as the living record and demonstration for Week 5 of the NoisiQ project.

By the end of this notebook you should be able to:
- See the upgraded Quirk-style circuit diagram with colored gate boxes by category
- See Pauli errors floating on qubit wires between gate boxes (not inside them)
- Use play/pause/step animation controls on error propagation
- Hover over any gate to see a tooltip with gate type, error info, and state transitions
- Verify the hover automatically shows different fields for single-shot vs many-shot runs
- Export an animation as GIF or HTML
- Confirm gate conjugation correctness identities pass

---

## Week 5 Milestone Summary

Week 5 has three parallel tracks: **visual polish** (Quirk-style diagram, corrected error
positioning on wires, animation controls, GIF/HTML export), the **hover tooltip system**
(seamless single/many-shot dispatch via `GateInfoExtractor`), and the **correctness harness**
(gate conjugation identity tests). `test_randomized.py` is moved to Week 6.

---

## Status Tracker

| Task | Owner | Status |
|------|-------|--------|
| `noisiq/visualization/theme.py` — finalize Quirk palette | TJ | ☐ Todo |
| `noisiq/visualization/circuit_diagram.py` — Quirk-style upgrade (colored gates, clean wires) | TJ | ☐ Todo |
| `noisiq/visualization/drawer.py` — errors float on wire between gate boxes | TJ | ☐ Todo |
| `noisiq/visualization/animation.py` — play/pause/step controls; many-shot shows highest-prob error | TJ | ☐ Todo |
| `noisiq/visualization/export.py` — `export_gif()` and `export_html()` | TJ | ☐ Todo |
| `noisiq/visualization/gate_info.py` — hover data extractor, single/many-shot dispatch | TJ | ☐ Todo |
| `noisiq/visualization/widgets.py` — `run_many()`, hover tooltip via `mpl_connect` + Output panel | TJ | ☐ Todo |
| `tests/backends/test_stim_correctness.py` — gate conjugation identities | TJ | ☐ Todo |
| All tests passing via `pytest` | TJ | ☐ Todo |
| CI passing on GitHub | TJ | ☐ Todo |
| Week 5 demo section of this notebook complete | TJ | ☐ Todo |

---

## File Build Order

```
1. noisiq/visualization/theme.py           ← Finalize if not done in Week 4
2. noisiq/visualization/circuit_diagram.py ← Upgrade (depends on theme.py)
3. noisiq/visualization/drawer.py          ← Error wire positioning fix
4. noisiq/visualization/animation.py       ← Enhanced animation (depends on drawer + theme)
5. noisiq/visualization/export.py          ← Export (depends on animation.py)
6. noisiq/visualization/gate_info.py       ← Hover data extractor (depends on AggregateResult + StimTableauResult)
7. noisiq/visualization/widgets.py         ← Hover integration (depends on gate_info.py)
8. tests/backends/test_stim_correctness.py ← Correctness tests (depends on stim_backend)
```

*Note: `tests/backends/test_randomized.py` moved to Week 6 to balance workload.*

---

## Technical Requirements for Week 5

**Error positioning on wires (drawer.py):**
- Pauli errors float on the qubit wire *between* gate boxes — never inside a gate box
- Error label pops up immediately after the gate that caused it
- Label rides the wire until the next gate box, then disappears; new error pops up after that gate
- For multi-qubit gates: error splits per propagation rules and rides both qubit lines forward

**Hover tooltip (gate_info.py + widgets.py):**
- `GateInfoExtractor.for_gate(op_idx, result) -> dict` dispatches on result type
- Single-shot fields: gate type, exact error (X/Y/Z/none), incoming state, ideal outgoing, actual outgoing
- Many-shot fields: gate type, error probability distribution (X/Y/Z/I %), most likely error, fidelity estimate
- Both modes expose a "Show propagation graph" button (fan-out — implemented Week 7)
- `widgets.py` detects last run type automatically; no manual flag needed

**Animation controls:**
- `ToggleButton`: play / pause
- `Button` (×2): step forward / step backward one frame
- `IntSlider`: animation speed (frames per second)
- All controls in horizontal `HBox` below the animation
- Many-shot animation: display highest-probability error at each gate with probability label

**Correctness identities to test:**
- `HXH† = Z`
- `HZH† = X`
- `SXS† = Y`

---

## Notes and Decisions Log

| Date | Note | Name |
|------|------|------|
| | | |

# Installation Instructions (Developer)

```bash
git clone https://github.com/dtsaxum-oss/noisiq.git
cd noisiq
pip install -e .
```

In [ ]:
import noisiq as nq

print(f"noisiq {nq.__version__} imported successfully")

# 1. Build a circuit and apply noise
circuit = nq.Circuit(n_qubits=3, name="ghz_noisy")
circuit.add_gate(nq.ir.H, qubits=[0])
circuit.add_gate(nq.ir.CNOT, qubits=[0, 1])
circuit.add_gate(nq.ir.CNOT, qubits=[1, 2])

noise = nq.noise.DepolarizingChannel(p=0.02)
noise_config = {i: noise.to_pauli_error() for i in range(len(circuit.operations))}

# 2. Display upgraded Quirk-style circuit diagram
print("\n--- Quirk-style circuit diagram ---")
nq.visualization.draw_circuit(circuit)

# 3. Single-shot: propagation animation with errors floating on wires
print("\n--- Single-shot propagation animation ---")
backend = nq.backends.StimTableauBackend()
single_result = backend.run_single_shot(circuit, noise_config=noise_config, seed=42)
anim = nq.visualization.animate_propagation(single_result, circuit)
# anim.show()  # ipywidgets panel: play/pause/step/speed controls

# 4. Many-shot: run via Visualizer; hover tooltip auto-switches to many-shot fields
print("\n--- Many-shot run + hover tooltip ---")
viz = nq.visualization.Visualizer(circuit)
viz.run_many(noise_config=noise_config, n_shots=500, seed=42)
# viz.show()   # interactive heatmap; hover over gate to see error probability distribution

# 5. Export animation
print("\n--- Exporting animation ---")
nq.visualization.export_gif(anim, path="week5_propagation.gif")
nq.visualization.export_html(anim, path="week5_propagation.html")
print("Exported week5_propagation.gif and week5_propagation.html")

In [ ]:
# Correctness check: HXH = Z
import numpy as np

H = nq.ir.H.matrix
X = nq.ir.X.matrix
Z = nq.ir.Z.matrix

result = H @ X @ H.conj().T
assert np.allclose(result, Z), f"HXH† should equal Z, got:\n{result}"
print("✓ HXH† = Z")

result2 = H @ Z @ H.conj().T
assert np.allclose(result2, X), f"HZH† should equal X, got:\n{result2}"
print("✓ HZH† = X")

print("\nAll conjugation identities verified.")